In [1]:
# %pip install openpyxl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge



df = pd.read_csv('../../Datasets/fromclass/house.csv')



# sns.histplot(x, kde=True)



In [2]:

# df.dtypes.value_counts()
# df = df.drop(['PoolQC','MiscFeature','Alley','Fence','MasVnrType','FireplaceQu'], axis=1)
# misssing_values = df.isnull().sum()
# misssing_values = misssing_values[misssing_values > 0].sort_values(ascending=False)
# misssing_values = (misssing_values / len(df)) *100
# misssing_values = pd.DataFrame({
#   'Missing values': misssing_values,
#   'Percentage': misssing_values.round()
# })
# misssing_values
# df.info()


In [3]:
# numerical_features =df.select_dtypes(include=['int64', 'float64']).columns.tolist()
# numerical_features.remove('Id')
# numerical_features.remove('SalePrice')

# print(f'Number of Numerical Features: {len(numerical_features)}')
# df[numerical_features].describe().T.sort_values(by='mean', ascending=False).head(10)
# numerical_correlation = df[numerical_features + ['SalePrice']].corr()['SalePrice'].sort_values(ascending=False)
# plt.figure(figsize=(12,8))
# sns.heatmap(
#     df[numerical_correlation.index[:11]].corr(),
#     cmap='coolwarm',
#     annot=True,
#     fmt=".2f",
#     vmin=-1,   # min correlation
#     vmax=1,    # max correlation
#     linewidths=0.5
# )

# plt.title('Correlation Heat Map: Top Numerical Features vs SalePrice')
# plt.xticks(rotation=45, ha='right')
# plt.tight_layout()
# plt.show()

In [4]:
# baseLine_features = ['OverallQual','GrLivArea','GarageCars','TotalBsmtSF','FullBath','YearBuilt']

# x = df[baseLine_features]
# y = df['SalePrice']
# lr = LinearRegression()
# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

# lr.fit(x_train, y_train)

# y_prep = lr.predict(x_test)
# r2 = r2_score(y_test, y_prep)

# coef_df = pd.DataFrame({
#   'Feature': baseLine_features,
#   'Coefficient': lr.coef_
# })
# input_data = pd.DataFrame([[7, 1710, 2, 856, 2, 2003]], 
#                           columns=baseLine_features)

# p=lr.predict(input_data)
# print(p)
# df[['OverallQual','GrLivArea','GarageCars','TotalBsmtSF','FullBath','YearBuilt','SalePrice']]





In [5]:

# baseLine_features = ['OverallQual','GrLivArea','GarageCars','TotalBsmtSF','FullBath','YearBuilt']
# x = df[baseLine_features]
# y = df['SalePrice']

# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42)

# lr = LinearRegression()
# lr.fit(x_train, y_train)

# y_prep = lr.predict(x_test)

# r2 = r2_score(y_test, y_prep)


# plt.scatter(y_test, y_prep, color='skyblue')
# plt.xlabel("Actual Price")
# plt.ylabel("Predicted Price")
# plt.grid()
# plt.show()
# r2

In [6]:
X = df.drop(['Id', 'SalePrice'], axis=1)
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


no_feature_cols = ['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                   'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                   'PoolQC', 'Fence', 'MiscFeature']

numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include=['object']).columns

for col in no_feature_cols:
  X_train[col] = X_train[col].fillna('None')
  X_test[col] = X_test[col].fillna('None')

for col in categorical_cols:
  if col not in no_feature_cols and X_train[col].isnull().sum () > 0:

    most_frequent = X_train[col].mode()[0]
    X_train[col] = X_train[col]. fillna(most_frequent)
    X_test[col] = X_test[col]. fillna(most_frequent)

misssing_values = X_train.isna().sum() # Missing Values sum
missing_percent = (misssing_values / len(X_train)) * 100 # Calculating missing values
missing_df = pd.DataFrame({ # Converting to DF
  'Missing values': misssing_values,
  'Percentage': missing_percent
})


missing_df = missing_df[missing_df['Missing values'] > 0].sort_values('Percentage', ascending=False) # Ascending the value
missing_df
df['LotFrontage']
df['MSSubClass'].mode()[0]


X_train['LotFrontage'] = X_train.groupby('Neighborhood')['LotFrontage'].transform(lambda x:x.fillna(x.median()))
X_test['LotFrontage'] = X_test.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
X_train.loc[X_train['GarageYrBlt'].isnull(), 'GarageYrBlt'] = X_train.loc[X_train['GarageYrBlt'].isnull(), 'YearBuilt']
X_test.loc[X_test['GarageYrBlt'].isnull(), 'GarageYrBlt'] = X_test.loc[X_test['GarageYrBlt'].isnull(), 'YearBuilt']
zero_cols = ['MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'GarageArea', 'GarageCars']

for col in numerical_cols:
  if col not in zero_cols and col != 'LotFrontage' and col != 'GarageYrBlt' and X_train[col].isnull().sum() > 0:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)




cat_cardinality = {col: X_train[col].nunique() for col in categorical_cols}
cat_cardinality_df = pd.DataFrame.from_dict(cat_cardinality, orient='index', columns=['Unique Values'])
cat_cardinality_df = cat_cardinality_df.sort_values('Unique Values', ascending=False)
cat_cardinality_df.head()



quality_mapping = {'None': 0, 'Po': 1,'Fa': 2,'TA': 3,'Gd': 4,'Ex': 5}
basement_exposure_mapping = {'None': 0,'No': 1,'Mn': 2,'Av': 3,'Gd': 4}
basement_finish_mapping = {'None': 0,'Unf': 1,'LWQ': 2,'Rec': 3,'BLQ': 4,'ALQ': 5,'GLQ': 6}
fence_mapping = {'None': 0,'Mnww': 1,'GdWo': 2,'MnPrv': 3,'GdPrv': 4}
garage_finish_mapping = {'None': 0,'Unf': 1,'RFn': 2,'Fin': 3}
lot_shape_mapping = {'Reg': 3,'IR1': 2,'IR2': 1,'IR3': 0}
functional_mapping = {'Sal': 0,'Sev': 1,'Maj2': 2,'Maj1': 3,'Mod': 4,'Min2': 5,'Min1': 6,'Typ': 7}

ordinal_features = {
    'ExterQual': quality_mapping,
    'ExterCond': quality_mapping,
    'BsmtQual': quality_mapping,
    'BsmtCond': quality_mapping,
    'BsmtExposure': basement_exposure_mapping,
    'BsmtFinType1': basement_finish_mapping,     # <-- FIXED NAME
    'BsmtFinType2': basement_finish_mapping,
    'HeatingQC': quality_mapping,
    'KitchenQual': quality_mapping,
    'FireplaceQu': quality_mapping,
    'GarageQual': quality_mapping,
    'GarageCond': quality_mapping,
    'GarageFinish': garage_finish_mapping,
    'PoolQC': quality_mapping,
    'Fence': fence_mapping,
    'LotShape': lot_shape_mapping,
    'Functional': functional_mapping,
    'CentralAir': {'N': 0, 'Y': 1}
}

# Apply mappings safely
for feature, mapping in ordinal_features.items():
        X_train[feature] = X_train[feature].map(mapping)
        X_test[feature] = X_test[feature].map(mapping)



nominal_features = [
'MSZoning', 'Street', 'Alley', 'LandContour', 'LotConfig', 'Neighborhood',
'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMat]',
'Exteriorist', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'Electrical',
'GarageType', 'MiscFeature', 'SaleType', 'SaleCondition', 'Utilities', 'LandSlope'
]
nominal_features = [f for f in nominal_features if f in X_train.columns]



for feature in nominal_features:
    dummies_train = pd.get_dummies(X_train[feature], prefix = feature, drop_first=True)
    dummies_test = pd.get_dummies(X_test[feature], prefix = feature, drop_first=True)

X_train


for col in dummies_train.columns:
  if col not in dummies_test.columns:
    dummies_test[col] = 0

for col in dummies_test.columns:
    if col not in dummies_train.columns:
        dummies_train[col] = 0
X_train = pd.concat([X_train, dummies_train], axis=1)
X_test = pd.concat([X_test, dummies_test], axis=1)

train_missing = X_train.isnull().sum().sum()
test_missing = X_test.isnull().sum().sum()

print(f"Missing values in training set: {train_missing}")
print(f"Missing values in test set: {test_missing}")


Missing values in training set: 94
Missing values in test set: 45
